# 02 · RAG Indexing
**Medical chatbot MVP**

This notebook:
1. Loads the processed documents from phase 1
2. Embeds them with `text-embedding-3-small`
3. Creates and saves a FAISS index
4. Tests retrieval with example queries
5. **Benchmarks query rewrite + LLM rerank** (pipeline accuracy)
6. Visualises the embedding space with t-SNE


In [2]:
import sys
import json
import pickle
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import faiss
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

sys.path.insert(0, str(Path('..').resolve()))

BASE_DIR = Path('..').resolve()
DOCUMENTS_PATH = BASE_DIR / 'data' / 'processed' / 'rag_documents.json'
INDEX_DIR = BASE_DIR / 'data' / 'processed' / 'faiss_index'
INDEX_DIR.mkdir(parents=True, exist_ok=True)

client = OpenAI()
print('OpenAI client OK')
print(f'Documents path: {DOCUMENTS_PATH}')
print(f'Index dir: {INDEX_DIR}')

OpenAI client OK
Documents path: D:\Medical Chatbot MVP\data\processed\rag_documents.json
Index dir: D:\Medical Chatbot MVP\data\processed\faiss_index


In [3]:
with open(DOCUMENTS_PATH, encoding='utf-8') as f:
    documents = json.load(f)

df = pd.DataFrame([{
    'id': d['id'],
    'type': d['metadata']['type'],
    'specialty': d['metadata'].get('specialty', '—'),
    'chars': len(d['text']),
} for d in documents])

print(f'Total documents: {len(documents)}')
print()
print(df.groupby('type').agg(count=('id','count'), avg_chars=('chars','mean')).round(0))

Total documents: 69

                count  avg_chars
type                            
doctor             18      563.0
knowledge_base     39      159.0
specialty          12      643.0


In [4]:

for doc_type in ['doctor', 'specialty', 'knowledge_base']:
    sample = next(d for d in documents if d['metadata']['type'] == doc_type)
    print(f'=== {doc_type.upper()} ===')
    print(sample['text'][:400])
    print('...' if len(sample['text']) > 400 else '')
    print()

=== DOCTOR ===
Лекар: доцент д-р Мария Иванова
Специалност: кардиолог
Опит: 18 години
Образование: Медицински университет — София. Специализация по кардиология — Виена, Австрия.

Доц. д-р Мария Иванова е кардиолог с 18-годишен опит в диагностика и лечение на сърдечно-съдови заболявания. Специализирала е инвазивна кардиология в Medizinische Universität Wien. Извършва над 500 коронарографии годишно. Говори свободн
...

=== SPECIALTY ===
Специалност: кардиолог
Специалист по сърдечно-съдови заболявания. Диагностицира и лекува проблеми със сърцето, артериите и вените.

При кои симптоми трябва да се запише час при кардиолог:
болка в гърдите, стягане в гърдите, задух при усилие, задух в покой, сърцебиене, неравномерен пулс, замайване, припадък, подуване на краката, умора при минимално усилие, високо кръвно налягане, ниско кръвно наляг
...

=== KNOWLEDGE_BASE ===
Адрес: бул. Васил Левски 47, София 1000
Телефон за записване: 02 800 12 34
Имейл: info@zdraveplus.bg
Уебсайт: www.zdraveplus.bg




In [5]:
EMBEDDING_MODEL = 'text-embedding-3-small'
EMBEDDING_DIM = 1536
BATCH_SIZE = 50

def embed_texts(texts: list[str]) -> np.ndarray:
    all_embeddings = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i : i + BATCH_SIZE]
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
        all_embeddings.extend([item.embedding for item in response.data])
        print(f'{min(i+BATCH_SIZE, len(texts))}/{len(texts)} embedded')
    return np.array(all_embeddings, dtype=np.float32)

texts = [d['text'] for d in documents]
embeddings = embed_texts(texts)
print(f'\Result: {embeddings.shape}  (documents X dimensions)')

50/69 embedded
69/69 embedded
\Result: (69, 1536)  (documents X dimensions)


In [6]:

np.save(INDEX_DIR / 'embeddings_raw.npy', embeddings)

In [7]:
emb_normalized = embeddings.copy()
faiss.normalize_L2(emb_normalized)

index = faiss.IndexFlatIP(EMBEDDING_DIM)
index.add(emb_normalized)

print(f'FAISS index: {index.ntotal} vectors, {EMBEDDING_DIM}D')
print(f'Type: {type(index).__name__}')

FAISS index: 69 vectors, 1536D
Type: IndexFlatIP


In [8]:
metadata = [{'id': d['id'], **d['metadata']} for d in documents]

index_bytes = faiss.serialize_index(index)
with open(INDEX_DIR / 'index.faiss', 'wb') as f:
    f.write(index_bytes)

with open(INDEX_DIR / 'metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

texts_map = {d['id']: d['text'] for d in documents}
with open(INDEX_DIR / 'texts.pkl', 'wb') as f:
    pickle.dump(texts_map, f)

np.save(INDEX_DIR / 'embeddings_raw.npy', embeddings)

for p in sorted(INDEX_DIR.iterdir()):
    print(f'{p.name}  {p.stat().st_size/1024:.1f} KB')

embeddings_raw.npy  414.1 KB
index.faiss  414.0 KB
metadata.pkl  4.9 KB
texts.pkl  43.5 KB


In [9]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from src.rag.retriever import MedicalRetriever

retriever = MedicalRetriever()

def search(query, top_k=5, doc_type=None, verbose=False,
           use_rewrite=True, use_rerank=True):    
    results = retriever.search(
        query, top_k=top_k, doc_type=doc_type,
        rewrite=use_rewrite, rerank=use_rerank,  
    )
    if verbose:
        print(f"[rewrite+rerank] returned {len(results)} results")
    return pd.DataFrame(results).fillna("-")


Retriever loaded: 69 documents in the index


In [10]:

query = 'Имам болки в гърдите и задух'
print(f'Query: "{query}"\n')
search(query, verbose=True, top_k=4)

Query: "Имам болки в гърдите и задух"

[rewrite] 'Имам болки в гърдите и задух'
 → 'Болки в гърдите и задух, кардиология, сърдечни заболявания, респираторни заболявания.'
[rewrite+rerank] returned 4 results


,vector_score,id,type,specialty,specialty_id,doctor_id,accepts_nhif,price,rating,rerank_score
0,0.648919,specialty_cardiology,specialty,кардиолог,cardiology,-,-,-,-,1.0
1,0.525061,specialty_pulmonology,specialty,пулмолог,pulmonology,-,-,-,-,1.0
2,0.496775,doctor_dr_001,doctor,кардиолог,cardiology,dr_001,False,100.0,4.9,1.0
3,0.468797,doctor_dr_002,doctor,кардиолог,cardiology,dr_002,True,80.0,4.7,1.0


In [11]:

query = 'силно главоболие и изтръпване на ръцете'
print(f'Query: "{query}"\n')
search(query, verbose=True, top_k=4)

Query: "силно главоболие и изтръпване на ръцете"

[rewrite] 'силно главоболие и изтръпване на ръцете'
 → 'главоболие, неврология, изтръпване на крайниците, неврологични разстройства'
[rewrite+rerank] returned 4 results


,vector_score,id,type,specialty,specialty_id,doctor_id,accepts_nhif,price,rating,rerank_score
0,0.547677,specialty_neurology,specialty,невролог,neurology,-,-,-,-,1.0
1,0.406540,doctor_dr_004,doctor,невролог,neurology,dr_004,True,90.0,4.6,1.0
2,0.404275,doctor_dr_003,doctor,невролог,neurology,dr_003,False,120.0,4.9,1.0
3,0.442505,kb_services_003,knowledge_base,-,-,-,-,-,-,0.5


In [12]:

query = 'кардиолог с опит в аритмии'
print(f'Query: "{query}" (само лекари)\n')
search(query, verbose=True, top_k=3, doc_type='doctor')

Query: "кардиолог с опит в аритмии" (само лекари)

[rewrite] 'кардиолог с опит в аритмии'
 → 'кардиолог, специализирал в диагностика и лечение на аритмии, сърдечни заболявания.'
[rewrite+rerank] returned 3 results


,vector_score,id,type,specialty,specialty_id,doctor_id,accepts_nhif,price,rating,rerank_score
0,0.547342,doctor_dr_001,doctor,кардиолог,cardiology,dr_001,False,100,4.9,1.0
1,0.481271,doctor_dr_002,doctor,кардиолог,cardiology,dr_002,True,80,4.7,1.0
2,0.392644,doctor_dr_017,doctor,уролог,urology,dr_017,False,110,4.9,0.0


In [13]:

query = 'до колко часа работи центърът'
print(f'Query: "{query}" (само knowledge base)\n')
search(query, verbose=True, top_k=3, doc_type='knowledge_base')

Query: "до колко часа работи центърът" (само knowledge base)

[rewrite] 'до колко часа работи центърът'
 → 'Какво е работното време на медицинския център?'
[rewrite+rerank] returned 3 results


,vector_score,id,type,specialty,specialty_id,doctor_id,accepts_nhif,price,rating,rerank_score
0,0.547757,kb_faq_014,knowledge_base,-,-,-,-,-,-,1.0
1,0.541147,kb_center_info_002,knowledge_base,-,-,-,-,-,-,0.9
2,0.441450,kb_center_info_010,knowledge_base,-,-,-,-,-,-,0.8


In [14]:

query = 'има ли лекар, който приема по НЗОК'
print(f'Query: "{query}"\n')
search(query, verbose=True, top_k=5)

Query: "има ли лекар, който приема по НЗОК"

[rewrite] 'има ли лекар, който приема по НЗОК'
 → 'Има ли лекар, който работи с НЗОК?'
[rerank] fallback to vector order — Reranker returned 21 scores for 20 candidates
[rewrite+rerank] returned 5 results


,vector_score,id,type,specialty,specialty_id,doctor_id,accepts_nhif,price,rating,rerank_score
0,0.705340,kb_center_info_007,knowledge_base,-,-,-,-,-,-,0.705340
1,0.531095,kb_faq_001,knowledge_base,-,-,-,-,-,-,0.531095
2,0.484576,kb_faq_013,knowledge_base,-,-,-,-,-,-,0.484576
3,0.471872,doctor_dr_012,doctor,ендокринолог,endocrinology,dr_012,True,80.0,4.7,0.471872
4,0.463269,doctor_dr_014,doctor,УНГ лекар,ent,dr_014,True,70.0,4.6,0.463269


In [15]:

query = 'внезапна парализа на лицето и загуба на реч'
print(f'Query: "{query}"\n')
search(query, verbose=True, top_k=4)

Query: "внезапна парализа на лицето и загуба на реч"

[rewrite] 'внезапна парализа на лицето и загуба на реч'
 → 'внезапна парализа на лицето и загуба на реч, неврология, инсулт, мозъчни заболявания'
[rewrite+rerank] returned 4 results


,vector_score,id,type,specialty,specialty_id,doctor_id,accepts_nhif,price,rating,rerank_score
0,0.541612,specialty_neurology,specialty,невролог,neurology,-,-,-,-,1.0
1,0.373060,doctor_dr_003,doctor,невролог,neurology,dr_003,False,120.0,4.9,1.0
2,0.352162,doctor_dr_004,doctor,невролог,neurology,dr_004,True,90.0,4.6,1.0
3,0.400120,kb_center_info_008,knowledge_base,-,-,-,-,-,-,0.0


## Pipeline accuracy benchmark (baseline vs rewrite+rerank)


In [16]:
test_queries = [
    ('болка в гърдите и задух', 'кардиолог'),
    ('главоболие и изтръпване', 'невролог'),
    ('болка в коляното след спорт', 'ортопед'),
    ('обрив и сърбеж по кожата', 'дерматолог'),
    ('киселини и стомашна болка', 'гастроентеролог'),
    ('умора и наддаване на тегло', 'ендокринолог'),
    ('хронична кашлица и задух', 'пулмолог'),
    ('болка в ухото и намален слух', 'УНГ лекар'),
    ('нередовен цикъл и болка в корема', 'гинеколог'),
    ('замъглено зрение', 'офталмолог'),
    ('парене при уриниране и кръв в урина', 'уролог'),
    ('депресия и безсъние', 'психиатър'),
]

def run_benchmark(use_rewrite: bool, use_rerank: bool) -> pd.DataFrame:
    rows = []
    for query, expected in test_queries:
        df = search(query, top_k=1, doc_type='specialty',
                    use_rewrite=use_rewrite, use_rerank=use_rerank)
        if df.empty:
            retrieved, score = '—', 0.0
        else:
            retrieved = df.iloc[0]['specialty']
            score = float(df.iloc[0].get('rerank_score',
                          df.iloc[0]['vector_score'])) 
        rows.append({
            'query': query[:38],
            'expected': expected,
            'retrieved': retrieved,
            'score': round(score, 4),
            'correct': expected.lower() in retrieved.lower(),
        })
    return pd.DataFrame(rows)

print('Running baseline (no rewrite, no rerank)...')
df_baseline = run_benchmark(use_rewrite=False, use_rerank=False)
acc_baseline = df_baseline['correct'].mean()

print('\nRunning full pipeline (rewrite + rerank)...')
df_pipeline = run_benchmark(use_rewrite=True, use_rerank=True)
acc_pipeline = df_pipeline['correct'].mean()

print(f'\nBaseline accuracy: {acc_baseline:.0%} ({df_baseline["correct"].sum()}/{len(df_baseline)})')
print(f'Pipeline accuracy: {acc_pipeline:.0%} ({df_pipeline["correct"].sum()}/{len(df_pipeline)})')
print(f'Improvement: +{(acc_pipeline - acc_baseline):.0%}')

df_comparison = df_baseline[['query','expected','retrieved','score','correct']].copy()
df_comparison.columns = ['query','expected','baseline_retrieved','baseline_score','baseline_correct']
df_comparison['pipeline_retrieved'] = df_pipeline['retrieved'].values
df_comparison['pipeline_score'] = df_pipeline['score'].values
df_comparison['pipeline_correct'] = df_pipeline['correct'].values
df_comparison


Running baseline (no rewrite, no rerank)...

Running full pipeline (rewrite + rerank)...
[rewrite] 'болка в гърдите и задух'
 → 'болка в гърдите, задух, кардиология, сърдечни заболявания, респираторни заболявания'
[rewrite] 'главоболие и изтръпване'
 → 'главоболие, неврология, изтръпване на крайниците, неврологични разстройства'
[rewrite] 'болка в коляното след спорт'
 → 'болка в коляното след спорт, ортопедия, спортна медицина, травми на опорно-двигателния апарат'
[rewrite] 'обрив и сърбеж по кожата'
 → 'дерматологични заболявания, обрив, сърбеж, дерматология'
[rewrite] 'киселини и стомашна болка'
 → 'киселини в стомаха, гастрит, стомашно-чревни заболявания'
[rewrite] 'умора и наддаване на тегло'
 → 'умора и наддаване на тегло, ендокринология, хормонални нарушения, метаболитни заболявания'
[rewrite] 'хронична кашлица и задух'
 → 'хронична кашлица, задух, пулмология, респираторни заболявания'
[rewrite] 'болка в ухото и намален слух'
 → 'болка в ухото, намален слух, оториноларингология,

,query,expected,baseline_retrieved,baseline_score,baseline_correct,pipeline_retrieved,pipeline_score,pipeline_correct
0,болка в гърдите и задух,кардиолог,пулмолог,0.4273,False,кардиолог,0.9,True
1,главоболие и изтръпване,невролог,невролог,0.3977,True,невролог,1.0,True
2,болка в коляното след спорт,ортопед,ортопед,0.3396,True,ортопед,1.0,True
3,обрив и сърбеж по кожата,дерматолог,дерматолог,0.3493,True,дерматолог,1.0,True
4,киселини и стомашна болка,гастроентеролог,ортопед,0.3643,False,гастроентеролог,1.0,True
5,умора и наддаване на тегло,ендокринолог,—,0.0000,False,ендокринолог,0.8,True
6,хронична кашлица и задух,пулмолог,пулмолог,0.3020,True,пулмолог,1.0,True
7,болка в ухото и намален слух,УНГ лекар,пулмолог,0.4181,False,УНГ лекар,1.0,True
8,нередовен цикъл и болка в корема,гинеколог,уролог,0.3514,False,гинеколог,0.9,True
9,замъглено зрение,офталмолог,офталмолог,0.3523,True,офталмолог,1.0,True


In [17]:
fig = go.Figure()

queries_short = df_comparison['query'].tolist()

fig.add_trace(go.Bar(
    name='Baseline',
    x=queries_short,
    y=df_comparison['baseline_score'],
    marker_color=[
        '#2ecc71' if c else '#e74c3c'
        for c in df_comparison['baseline_correct']
    ],
    opacity=0.5,
))

fig.add_trace(go.Bar(
    name='Rewrite + Rerank',
    x=queries_short,
    y=df_comparison['pipeline_score'],
    marker_color=[
        '#27ae60' if c else '#c0392b'
        for c in df_comparison['pipeline_correct']
    ],
))

fig.update_layout(
    barmode='group',
    title='Baseline vs Rewrite+Rerank — similarity score per query (green = correct)',
    xaxis_title='Query',
    yaxis_title='Score',
    xaxis_tickangle=-35,
    legend_title='Pipeline',
    height=500,
)
fig.show()


## Visualisation of embedding space (t-SNE)

Reducing from 1536D → 2D to inspect document clustering.
Well-separated specialty clusters indicate the embedding space supports retrieval.
With only 69 documents, some overlap is expected — enriching specialty docs with
symptom keywords will tighten clusters.


In [18]:
from sklearn.manifold import TSNE

reducer = TSNE(
    n_components=2,
    random_state=42,
    perplexity=30, # controls local vs global structure (try 5–50)
    learning_rate='auto',
    init='pca', # PCA init is more stable than random
    max_iter=1000,
)
coords = reducer.fit_transform(emb_normalized)
method = 't-SNE'

print(f'Reduction with {method}: {emb_normalized.shape} → {coords.shape}')

Reduction with t-SNE: (69, 1536) → (69, 2)


In [19]:
df_plot = pd.DataFrame({
    'x': coords[:, 0],
    'y': coords[:, 1],
    'type': [m.get('type') for m in metadata],
    'specialty': [m.get('specialty', m.get('source', m.get('type', '—'))) for m in metadata],
    'id': [m.get('id') for m in metadata],
})

fig = px.scatter(
    df_plot, x='x', y='y',
    color='specialty',
    symbol='type',
    hover_data=['id', 'type'],
    title=f'Embedding dimension ({method}) — {len(documents)} documents',
    width=900, height=600,
)
fig.update_traces(marker_size=8)
fig.show()

## Totals

In [20]:
print('=' * 50)
print('RAG INDEXING SUMMARY')
print('=' * 50)
print(f'Model: {EMBEDDING_MODEL}')
print(f'Dimensions: {EMBEDDING_DIM}')
print(f'Documents: {index.ntotal}')
print(f'FAISS type: IndexFlatIP (cosine)')
print()
print(f'Baseline accuracy: {acc_baseline:.0%}')
print(f'Pipeline accuracy: {acc_pipeline:.0%}  (rewrite + rerank)')
print()
print('Files:')
for f in sorted(INDEX_DIR.iterdir()):
    print(f'{f.name:<25} {f.stat().st_size/1024:.1f} KB')
print()


RAG INDEXING SUMMARY
Model: text-embedding-3-small
Dimensions: 1536
Documents: 69
FAISS type: IndexFlatIP (cosine)

Baseline accuracy: 58%
Pipeline accuracy: 100%  (rewrite + rerank)

Files:
embeddings_raw.npy        414.1 KB
index.faiss               414.0 KB
metadata.pkl              4.9 KB
texts.pkl                 43.5 KB

